In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# ==================================================
# 1) CARREGAR E PREPARAR OS DADOS
# ==================================================
iris = load_iris()
X = iris.data   # 4 atributos
y = iris.target # 3 classes

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
    shuffle=True,
)

# Normalizacao dos atributos
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# One-hot encoding das classes
num_classes = 3
y_train_oh = np.eye(num_classes)[y_train]
y_test_oh = np.eye(num_classes)[y_test]

In [3]:
# ==================================================
# 2) FUNCOES DE ATIVACAO E LOSS
# ==================================================
def relu(x):
    return np.maximum(0.0, x)


def relu_derivative(x):
    return (x > 0.0).astype(float)


def softmax(z):
    # Softmax estavel para evitar overflow
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def cross_entropy(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

In [4]:
# ==================================================
# 3) ARQUITETURA DA REDE
# Entrada: 4 neuronios
# Oculta: 10 neuronios (entre 5 e 10)
# Saida: 3 neuronios (softmax)
# ==================================================
np.random.seed(42)
input_size = 4
hidden_size = 10
output_size = 3

# Inicializacao He (adequada para ReLU)
W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
b2 = np.zeros((1, output_size))

In [5]:
# ==================================================
# 4) FORWARD PASS E BACKPROPAGATION
# ==================================================
def forward_pass(X, W1, b1, W2, b2):
    # Camada oculta
    z1 = X @ W1 + b1
    a1 = relu(z1)

    # Camada de saida
    z2 = a1 @ W2 + b2
    y_pred = softmax(z2)

    return z1, a1, z2, y_pred


def backward_pass(X, y_true, z1, a1, y_pred, W2):
    m = X.shape[0]

    # Gradiente na saida (softmax + cross-entropy)
    dz2 = (y_pred - y_true) / m
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # Propagacao para camada oculta
    da1 = dz2 @ W2.T
    dz1 = da1 * relu_derivative(z1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2

In [6]:
# ==================================================
# 5) TREINAMENTO E PREDICAO
# ==================================================
def train(
    X_train,
    y_train_oh,
    W1,
    b1,
    W2,
    b2,
    epochs=1000,
    lr=0.03,
):
    loss_history = []
    n = X_train.shape[0]

    for epoch in range(epochs):
        # Embaralha as amostras em cada epoca
        idx = np.random.permutation(n)
        X_shuffled = X_train[idx]
        y_shuffled = y_train_oh[idx]

        total_loss = 0.0

        # Atualizacao amostra a amostra (SGD)
        for i in range(n):
            x_i = X_shuffled[i:i + 1]
            y_i = y_shuffled[i:i + 1]

            # Forward
            z1, a1, z2, y_pred = forward_pass(x_i, W1, b1, W2, b2)

            # Loss
            total_loss += cross_entropy(y_i, y_pred)

            # Backward
            dW1, db1, dW2, db2 = backward_pass(x_i, y_i, z1, a1, y_pred, W2)

            # Atualizacao dos parametros
            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2

        avg_loss = total_loss / n
        loss_history.append(avg_loss)

    return W1, b1, W2, b2, loss_history


def predict(X, W1, b1, W2, b2):
    _, _, _, y_pred = forward_pass(X, W1, b1, W2, b2)
    return np.argmax(y_pred, axis=1)

In [7]:
# ==================================================
# 6) EXECUCAO FINAL E AVALIACAO
# ==================================================
W1, b1, W2, b2, loss_history = train(
    X_train,
    y_train_oh,
    W1,
    b1,
    W2,
    b2,
    epochs=1000,
    lr=0.03,
)

y_pred_test = predict(X_test, W1, b1, W2, b2)
accuracy_test = np.mean(y_pred_test == y_test)

# Evolucao da loss em texto (de 1 em 1)
print("Evolucao da loss (de 1 em 1):")
for epoca, loss in enumerate(loss_history, start=1):
    print(f"Epoca {epoca}/1000 - Loss: {loss:.6f}")

print(f"\nAcuracia final no conjunto de teste: {accuracy_test * 100:.2f}%")

Evolucao da loss (de 1 em 1):
Epoca 1/1000 - Loss: 1.033879
Epoca 2/1000 - Loss: 0.822264
Epoca 3/1000 - Loss: 0.628697
Epoca 4/1000 - Loss: 0.510213
Epoca 5/1000 - Loss: 0.430939
Epoca 6/1000 - Loss: 0.380348
Epoca 7/1000 - Loss: 0.335748
Epoca 8/1000 - Loss: 0.289411
Epoca 9/1000 - Loss: 0.264108
Epoca 10/1000 - Loss: 0.235103
Epoca 11/1000 - Loss: 0.204807
Epoca 12/1000 - Loss: 0.187793
Epoca 13/1000 - Loss: 0.175441
Epoca 14/1000 - Loss: 0.161101
Epoca 15/1000 - Loss: 0.150244
Epoca 16/1000 - Loss: 0.145081
Epoca 17/1000 - Loss: 0.129027
Epoca 18/1000 - Loss: 0.128939
Epoca 19/1000 - Loss: 0.121815
Epoca 20/1000 - Loss: 0.114986
Epoca 21/1000 - Loss: 0.110930
Epoca 22/1000 - Loss: 0.105368
Epoca 23/1000 - Loss: 0.101339
Epoca 24/1000 - Loss: 0.103212
Epoca 25/1000 - Loss: 0.095100
Epoca 26/1000 - Loss: 0.091372
Epoca 27/1000 - Loss: 0.089430
Epoca 28/1000 - Loss: 0.090295
Epoca 29/1000 - Loss: 0.088011
Epoca 30/1000 - Loss: 0.084343
Epoca 31/1000 - Loss: 0.075913
Epoca 32/1000 - Lo